# 06. Leave-one-borehole-out evaluation

This notebook plots triangle quality, pinch-out boundary changes, and interface-elevation and thickness errors for the leave-one-borehole-out experiment. Each fold retains all XY locations and masks every interface elevation at one borehole; the example contains 39 folds.

**Input:** The evaluation tables, metadata, and full-model package generated by `run_evaluation.py`. The input directory is selected by `HG3D_EVALUATION_DIR` (default: `outputs/evaluation/`).

**Output:** `triangle_quality_metrics.png`, `front_comparison.png`, and `prediction_errors.png`. Figures are displayed and saved to `HG3D_FIGURE_DIR` (default: `outputs/figures/`).

**Interpretation:** The reference elevations include model-completed values and are not all independent measurements. Boundary changes measure sensitivity relative to the full reconstruction; they do not measure accuracy against an independently mapped geological boundary.


## 1. Imports and configuration


In [ ]:
import json
import os
from pathlib import Path

os.environ.setdefault("MPLCONFIGDIR", "/tmp/matplotlib-borehole-evaluation-plot")
Path(os.environ["MPLCONFIGDIR"]).mkdir(parents=True, exist_ok=True)

# Project paths.
PROJECT_ROOT = next(
    (
        path
        for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
        if (path / "run_model.py").is_file() and (path / "notebooks").is_dir()
    ),
    None,
)
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Start this notebook from the HarmonicGeo3D directory or a subdirectory."
    )

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from matplotlib.collections import LineCollection
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator

# Input data.
EVALUATION_DIR = Path(
    os.environ.get("HG3D_EVALUATION_DIR", PROJECT_ROOT / "outputs" / "evaluation")
)
REQUIRED = (
    "triangle_quality.csv",
    "front_overlay.csv",
    "masked_prediction_errors.csv",
    "masked_thickness_errors.csv",
    "interface_metrics.csv",
    "thickness_metrics.csv",
    "evaluation_metadata.json",
    "full/demo_model.npz",
)

CM_TO_INCH = 1 / 2.54
FONT_FAMILY = "Arial"
FULL_COLOR = "#FFC0C8"
FOLD_CMAP = sns.color_palette("crest", as_cmap=True)
TICK_INTERVAL_M = 200.0

SELECTED_BODY_NAMES = ("body-01", "body-02", "body-03")
LEGEND_GAP_CM = 0.20
RANDOM_SEED = 20260914
QUALITY_FIGURE_WIDTH_CM, QUALITY_FIGURE_HEIGHT_CM = 12.5, 9.0
# The boundary comparison contains one panel per selected geological unit.
FRONT_FIGURE_WIDTH_CM, FRONT_FIGURE_HEIGHT_CM = 14.5, 7.0
ERROR_FIGURE_WIDTH_CM, ERROR_FIGURE_HEIGHT_CM = 15.5, 11.5
PLOT_RC = {
    "font.family": FONT_FAMILY,
    "axes.unicode_minus": False,
    "figure.facecolor": "white",
    "axes.facecolor": "white",
}

# Figure output.
FIGURE_DIR = Path(
    os.environ.get("HG3D_FIGURE_DIR", PROJECT_ROOT / "outputs" / "figures")
)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

## 2. Data loading and validation


In [ ]:
missing = [
    str(EVALUATION_DIR / name)
    for name in REQUIRED
    if not (EVALUATION_DIR / name).is_file()
]
if missing:
    raise FileNotFoundError(
        "Evaluation files are missing; run run_evaluation.py first: "
        + ", ".join(missing)
    )
metadata = json.loads(
    (EVALUATION_DIR / "evaluation_metadata.json").read_text(encoding="utf-8")
)
quality_df = pd.read_csv(EVALUATION_DIR / "triangle_quality.csv")
front_df = pd.read_csv(EVALUATION_DIR / "front_overlay.csv")
interface_errors_df = pd.read_csv(EVALUATION_DIR / "masked_prediction_errors.csv")
thickness_errors_df = pd.read_csv(EVALUATION_DIR / "masked_thickness_errors.csv")
interface_metrics_df = pd.read_csv(EVALUATION_DIR / "interface_metrics.csv").set_index(
    "interface"
)
thickness_metrics_df = pd.read_csv(EVALUATION_DIR / "thickness_metrics.csv").set_index(
    "body"
)
with np.load(
    EVALUATION_DIR / "full" / "demo_model.npz", allow_pickle=False
) as model_data:
    initial_nodes_xy = model_data["initial_nodes_xy"]
masked_folds = metadata["masked_drills"]
if len(masked_folds) != metadata["fold_count"] or set(
    quality_df.loc[quality_df.state == "mask", "fold"]
) != set(masked_folds):
    raise ValueError(
        "The evaluation tables must contain every borehole fold declared in the metadata."
    )

# Each borehole fold has the same colour in the quality and boundary figures.
fold_colors = {
    fold: FOLD_CMAP(i / max(1, len(masked_folds) - 1))
    for i, fold in enumerate(masked_folds)
}

## 3. Plot triangle quality by borehole


In [ ]:
with plt.rc_context(PLOT_RC):
    # Triangle-quality summaries are calculated separately for each reconstruction.
    quality_groups = [
        ("full", quality_df.loc[quality_df.state == "full", "quality"].to_numpy())
    ]
    quality_groups += [
        (
            fold,
            quality_df.loc[
                (quality_df.state == "mask") & (quality_df.fold == fold), "quality"
            ].to_numpy(),
        )
        for fold in masked_folds
    ]
    quality_statistics_by_fold = {}
    for fold, values in quality_groups:
        values = np.asarray(values, dtype=float)
        if not len(values) or not np.isfinite(values).all():
            raise ValueError(
                "Each reconstruction must contain nonempty, finite triangle-quality values."
            )
        quality_statistics_by_fold[fold] = {
            "mean": float(np.mean(values)),
            "p05": float(np.percentile(values, 5)),
            "minimum": float(np.min(values)),
        }
    full_statistics = quality_statistics_by_fold["full"]
    fold_statistics_df = pd.DataFrame.from_records(
        [{"fold": fold, **quality_statistics_by_fold[fold]} for fold in masked_folds]
    ).set_index("fold")

    y = np.arange(len(masked_folds))
    colors = [fold_colors[fold] for fold in masked_folds]
    fold_labels = list(masked_folds)
    fig, axes = plt.subplots(
        1,
        3,
        figsize=(
            QUALITY_FIGURE_WIDTH_CM * CM_TO_INCH,
            QUALITY_FIGURE_HEIGHT_CM * CM_TO_INCH,
        ),
        dpi=600,
        sharey=True,
    )
    for ax, (metric, panel_label, xlabel) in zip(
        axes,
        (
            ("mean", "(a)", "Mean q (higher is better)"),
            ("p05", "(b)", "P05 q (higher is better)"),
            ("minimum", "(c)", "Minimum q (higher is better)"),
        ),
    ):
        values = fold_statistics_df[metric].to_numpy()
        baseline = full_statistics[metric]
        ax.hlines(y, baseline, values, color="0.80", linewidth=0.36, zorder=1)
        ax.scatter(
            values, y, s=10, c=colors, edgecolors="white", linewidths=0.5, zorder=3
        )
        ax.axvline(baseline, color=FULL_COLOR, linestyle="--", linewidth=0.5, zorder=2)
        baseline_text = f"Full = {baseline:.5f}"
        ax.text(
            0.02,
            0.98,
            panel_label,
            transform=ax.transAxes,
            ha="left",
            va="top",
            fontsize=7,
        )
        ax.text(
            0.98,
            0.98,
            baseline_text,
            transform=ax.transAxes,
            ha="right",
            va="top",
            fontsize=5,
            bbox=dict(facecolor="white", edgecolor="none", pad=0.3),
        )
        ax.set_xlabel(xlabel, fontsize=6)
        low, high = (
            min(float(values.min()), baseline),
            max(float(values.max()), baseline),
        )
        span = max(high - low, 0.001)
        ax.set_xlim(max(0, low - 0.15 * span), min(1, high + 0.15 * span))
        # The upper margin contains panel labels and full-reconstruction values.
        ax.set_ylim(len(masked_folds) - 0.5, -2.5)
        ax.set_yticks(y)
        ax.ticklabel_format(axis="x", style="plain", useOffset=False)
        ax.locator_params(axis="x", nbins=4)

        ax.tick_params(
            top=True, right=True, direction="in", labelsize=4, length=1.2, width=0.3
        )
        ax.grid(alpha=0.8, color="lightgray", linewidth=0.25)
        ax.grid(axis="y", alpha=0.35, linewidth=0.15)

        for spine in ax.spines.values():
            spine.set_linewidth(0.6)
            spine.set_color("#333333")

    axes[0].set_yticks(y, fold_labels, fontsize=4)
    axes[0].set_ylabel("Masked borehole", fontsize=6)

    fig.tight_layout(pad=0.15, w_pad=0.1)
    fig.savefig(FIGURE_DIR / "triangle_quality_metrics.png", dpi=600)
    plt.show()

## 4. Plot pinch-out boundary changes


In [ ]:
body_names = metadata["body_names"]
if len(body_names) != 5:
    raise ValueError(
        "The example overlay requires five geological units ordered from bottom to top."
    )
body_name_to_index = {name: index for index, name in enumerate(body_names)}
missing_bodies = [
    name for name in SELECTED_BODY_NAMES if name not in body_name_to_index
]
if missing_bodies:
    raise ValueError(
        f"The model is missing the selected geological units: {missing_bodies}"
    )
selected_body_indices = [body_name_to_index[name] for name in SELECTED_BODY_NAMES]
with np.load(
    EVALUATION_DIR / "full" / "demo_model.npz", allow_pickle=False
) as model_data:
    body_boundaries = model_data["body_boundaries"]
    if list(model_data["body_names"]) != body_names:
        raise ValueError(
            "The geological-unit order must match between the model and metadata."
        )

with plt.rc_context(PLOT_RC):
    x_padding = np.ptp(initial_nodes_xy[:, 0]) * 0.03
    y_padding = np.ptp(initial_nodes_xy[:, 1]) * 0.03
    x_limits = (
        initial_nodes_xy[:, 0].min() - x_padding,
        initial_nodes_xy[:, 0].max() + x_padding,
    )
    y_limits = (
        initial_nodes_xy[:, 1].min() - y_padding,
        initial_nodes_xy[:, 1].max() + y_padding,
    )
    fig, axes = plt.subplots(
        1,
        3,
        figsize=(
            FRONT_FIGURE_WIDTH_CM * CM_TO_INCH,
            FRONT_FIGURE_HEIGHT_CM * CM_TO_INCH,
        ),
        dpi=600,
        sharex=True,
        sharey=True,
    )
    for ax, body_index, letter in zip(axes, selected_body_indices, "abc"):
        subset = front_df.loc[front_df.body == body_index]

        for fold in masked_folds:
            rows = subset.loc[(subset.state == "mask") & (subset.fold == fold)]
            segments = rows[["x1", "y1", "x2", "y2"]].to_numpy().reshape(-1, 2, 2)
            ax.add_collection(
                LineCollection(
                    segments,
                    colors=[fold_colors[fold]],
                    linewidths=0.5,
                    alpha=1,
                    zorder=3,
                )
            )
        rows = subset.loc[subset.state == "full"]
        segments = rows[["x1", "y1", "x2", "y2"]].to_numpy().reshape(-1, 2, 2)
        ax.add_collection(
            LineCollection(
                segments,
                colors=[FULL_COLOR],
                linewidths=0.6,
                linestyles="dashed",
                zorder=5,
            )
        )

        ax.scatter(
            initial_nodes_xy[:, 0],
            initial_nodes_xy[:, 1],
            s=6,
            facecolor="white",
            edgecolor="#333333",
            linewidth=0.5,
            zorder=2,
        )
        ax.set_xlim(*x_limits)
        ax.set_ylim(*y_limits)
        ax.set_aspect("equal", adjustable="box")

        # Map styling follows the conformal-mesh figure.
        for spine in ax.spines.values():
            spine.set_linewidth(0.6)
            spine.set_color("#333333")
        ax.set_xlabel("X (m)", family=FONT_FAMILY, fontsize=6)
        if ax is axes[0]:
            ax.set_ylabel("Y (m)", family=FONT_FAMILY, fontsize=6)
        ax.set_axisbelow(True)
        ax.grid(axis="both", alpha=0.8, color="lightgray", linewidth=0.25)
        ax.xaxis.set_major_locator(MultipleLocator(TICK_INTERVAL_M))
        ax.yaxis.set_major_locator(MultipleLocator(TICK_INTERVAL_M))
        ax.tick_params(
            axis="x",
            which="both",
            top=True,
            bottom=True,
            labeltop=False,
            labelbottom=True,
            direction="in",
            labelsize=4,
            length=2,
            width=0.5,
        )
        ax.tick_params(
            axis="y",
            which="both",
            left=True,
            right=True,
            labelright=False,
            labelleft=ax is axes[0],
            direction="in",
            labelsize=4,
            length=2,
            width=0.5,
            rotation=90,
        )
        ax.ticklabel_format(style="plain", useOffset=False)
        plt.setp(ax.get_xticklabels(), family=FONT_FAMILY, ha="center", va="top")
        if ax is axes[0]:
            plt.setp(
                ax.get_yticklabels(),
                family=FONT_FAMILY,
                rotation=90,
                ha="center",
                va="center",
            )
        ax.text(
            0.02,
            0.98,
            f"({letter})",
            transform=ax.transAxes,
            ha="left",
            va="top",
            family=FONT_FAMILY,
            fontsize=7,
        )

    handles = [
        Line2D(
            [0],
            [0],
            color=FULL_COLOR,
            linestyle="--",
            linewidth=1,
            label="Full model",
        ),
        Line2D(
            [0],
            [0],
            marker="o",
            linestyle="None",
            markerfacecolor="white",
            markeredgecolor="#333333",
            markeredgewidth=0.5,
            markersize=np.sqrt(6),
            label="Boreholes",
        ),
    ]
    handles += [
        Line2D(
            [0],
            [0],
            color=fold_colors[fold],
            linewidth=1,
            label=fold,
        )
        for fold in masked_folds
    ]
    # The legend follows the rendered X-label bounds, leaving a fixed gap
    # below the equal-scale map panels.
    for ax in axes:
        ax.set_anchor("N")
    fig.subplots_adjust(left=0.045, right=0.995, bottom=0.24, top=0.985, wspace=0.08)
    fig.canvas.draw()
    renderer = fig.canvas.get_renderer()
    label_bottom = min(ax.get_tightbbox(renderer).y0 for ax in axes) / fig.bbox.height
    legend_top = label_bottom - LEGEND_GAP_CM / (fig.get_size_inches()[1] / CM_TO_INCH)
    front_legend = fig.legend(
        handles=handles,
        loc="upper center",
        bbox_to_anchor=(0.5, legend_top),
        ncol=14,
        frameon=False,
        fontsize=4,
        columnspacing=0.4,
        handlelength=2,
        borderaxespad=0,
    )
    fig.savefig(FIGURE_DIR / "front_comparison.png", dpi=600)
    plt.show()

## 5. Plot interface-elevation and thickness errors


In [ ]:
# Interface-elevation and thickness errors share the same signed-error limits.
all_errors = np.r_[
    interface_errors_df["signed_error"].to_numpy(),
    thickness_errors_df["signed_error"].to_numpy(),
]
finite_errors = all_errors[np.isfinite(all_errors)]
error_limit = max(5.0, float(np.ceil(np.max(np.abs(finite_errors)) * 1.12 / 5.0) * 5.0))

# Both panels in each column use bottom-to-top stratigraphic ordering.
error_interface_order = [f"layer-{i:02d}-bot" for i in range(5)] + ["Ground"]
error_body_order = [f"body-{i:02d}" for i in range(5)]
error_interface_labels = [
    "Coal base",
    "Coal–Lacus",
    "Lac–Delta",
    "Delta–Flu",
    "Flu–Topsoil",
    "Topsoil top",
]
error_body_labels = ["Coal", "Lacustrine", "Deltaic", "Fluvial", "Topsoil"]
error_groups = []
error_metrics = []
for errors_df, category, names, saved_metrics in (
    (interface_errors_df, "interface", error_interface_order, interface_metrics_df),
    (thickness_errors_df, "body", error_body_order, thickness_metrics_df),
):
    values = errors_df.pivot(
        index="drill", columns=category, values="signed_error"
    ).reindex(index=masked_folds, columns=names)
    if not np.isfinite(values.to_numpy()).all():
        raise ValueError(
            "Each interface and geological unit must have valid prediction errors for every fold."
        )
    metrics = saved_metrics.loc[names, ["MAE", "RMSE"]]
    computed = np.column_stack(
        (
            np.mean(np.abs(values.to_numpy()), axis=0),
            np.sqrt(np.mean(values.to_numpy() ** 2, axis=0)),
        )
    )
    if not np.allclose(metrics.to_numpy(), computed, rtol=1e-10, atol=1e-10):
        raise ValueError(
            "The MAE and RMSE summaries must match the per-borehole prediction errors."
        )
    error_groups.append(values)
    error_metrics.append(metrics)

metric_limit = float(
    np.ceil(max(frame.to_numpy().max() for frame in error_metrics) / 2.0) * 2.0
)
metric_colors = {"MAE": "#668ca6", "RMSE": "#b35c44"}
with plt.rc_context(PLOT_RC):
    fig, axes = plt.subplots(
        2,
        2,
        figsize=(
            ERROR_FIGURE_WIDTH_CM * CM_TO_INCH,
            ERROR_FIGURE_HEIGHT_CM * CM_TO_INCH,
        ),
        dpi=600,
        gridspec_kw={"width_ratios": [6, 5], "height_ratios": [1.1, 1]},
    )
    axes[0, 1].sharey(axes[0, 0])
    axes[1, 1].sharey(axes[1, 0])
    for column, (values, metrics, tick_labels) in enumerate(
        zip(error_groups, error_metrics, (error_interface_labels, error_body_labels))
    ):
        positions = np.arange(len(values.columns))
        top, bottom = axes[:, column]
        top.boxplot(
            [values[name].to_numpy() for name in values.columns],
            positions=positions,
            widths=0.5,
            whis=1.5,
            patch_artist=True,
            showfliers=False,
            boxprops={"facecolor": "#dce8f0", "edgecolor": "#1f5d8f", "linewidth": 0.5},
            medianprops={"color": "#b35c44", "linewidth": 0.8},
            whiskerprops={"color": "#1f5d8f", "linewidth": 0.5},
            capprops={"color": "#1f5d8f", "linewidth": 0.5},
        )
        jitter = np.random.default_rng(RANDOM_SEED).uniform(
            -0.15, 0.15, len(masked_folds)
        )
        for position, name in zip(positions, values.columns):
            top.scatter(
                position + jitter,
                values[name],
                s=5,
                color="#668ca6",
                alpha=0.42,
                edgecolors="none",
                zorder=3,
            )
        top.axhline(0, color="0.35", linestyle="--", linewidth=0.5, zorder=1)
        top.set_ylim(-error_limit, error_limit)
        width = 0.34
        for offset, metric in ((-0.5, "MAE"), (0.5, "RMSE")):
            bottom.bar(
                positions + offset * width,
                metrics[metric],
                width=width,
                color=metric_colors[metric],
                edgecolor="white",
                linewidth=0.35,
                label=metric,
                zorder=3,
            )
        bottom.set_ylim(0, metric_limit)
        bottom.yaxis.set_major_locator(MultipleLocator(2))
        for row, ax in enumerate((top, bottom)):
            ax.set_xticks(positions, tick_labels, fontsize=5.5)
            ax.set_xlim(-0.6, len(positions) - 0.4)
            ax.set_axisbelow(True)
            ax.grid(axis="y", alpha=0.8, color="lightgray", linewidth=0.25)
            for spine in ax.spines.values():
                spine.set_linewidth(0.6)
                spine.set_color("#333333")
            ax.tick_params(
                top=True,
                right=True,
                direction="in",
                labelsize=5.5,
                length=2,
                width=0.5,
                labelleft=column == 0,
            )
            ax.ticklabel_format(axis="y", style="plain", useOffset=False)
            ax.text(
                0.02,
                0.98,
                ("(a)", "(b)", "(c)", "(d)")[2 * row + column],
                transform=ax.transAxes,
                ha="left",
                va="top",
                fontsize=7,
            )
    axes[0, 0].set_ylabel("Prediction - reference (m)", fontsize=6.5)
    axes[1, 0].set_ylabel("MAE / RMSE (m)", fontsize=6.5)
    distribution_legend = axes[0, 1].legend(
        handles=[
            Patch(
                facecolor="#dce8f0", edgecolor="#1f5d8f", linewidth=0.5, label="25–75%"
            ),
            Line2D([0], [0], color="#b35c44", linewidth=1, label="Median"),
            Line2D(
                [0],
                [0],
                marker="o",
                linestyle="None",
                color="#668ca6",
                alpha=0.5,
                markersize=3,
                label=f"{len(masked_folds)} masked predictions",
            ),
            Line2D(
                [0],
                [0],
                color="0.35",
                linestyle="--",
                linewidth=0.5,
                label="Zero error",
            ),
        ],
        loc="upper right",
        borderaxespad=0.2,
        frameon=True,
        framealpha=0.9,
        edgecolor="0.5",
        fontsize=5,
        handlelength=1.8,
    )
    metric_legend = axes[1, 1].legend(
        loc="upper right",
        ncol=2,
        borderaxespad=0.25,
        fontsize=5.5,
        frameon=True,
        framealpha=0.9,
        edgecolor="0.5",
        handlelength=1.6,
    )
    for legend in (distribution_legend, metric_legend):
        legend.get_frame().set_linewidth(0.4)
    fig.tight_layout(pad=0.6, w_pad=0.7, h_pad=1.25)
    figure_dir = FIGURE_DIR
    figure_dir.mkdir(exist_ok=True)
    fig.savefig(figure_dir / "prediction_errors.png", dpi=600)
    plt.show()